# Random Forest From Scratch

**What you'll build:** Bagging over DecisionTree with feature subsampling.

**Prerequisites:** [decision-tree.ipynb](decision-tree.ipynb) — the DecisionTree class is reused here.

## Concept Recap

**Bagging (Bootstrap Aggregating):**
- Sample n training points with replacement (bootstrap sample)
- Train a tree on each bootstrap sample
- Aggregate predictions by majority vote (classification) or mean (regression)

**Why it reduces variance:**
- Each tree sees a different bootstrap sample → different trees
- Errors of individual trees are uncorrelated
- Averaging uncorrelated noisy predictions cancels the noise

**Feature subsampling:** at each split, only consider √d features (not all d).
This further de-correlates trees — if one strong feature dominates, it won't dominate every tree.

**OOB (Out-of-Bag) score:** ~37% of samples not in each bootstrap. Use them as a free validation set.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

# Paste DecisionTree class from decision-tree.ipynb (or import if available)
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature; self.threshold = threshold
        self.left = left; self.right = right; self.value = value

class DecisionTree:
    def __init__(self, max_depth=5, min_samples_split=2, max_features=None):
        self.max_depth = max_depth; self.min_samples_split = min_samples_split
        self.max_features = max_features; self.root = None

    def fit(self, X, y): self.root = self._grow(X, y, 0)
    def predict(self, X): return np.array([self._traverse(x, self.root) for x in X])

    def _gini(self, y):
        _, counts = np.unique(y, return_counts=True)
        p = counts / len(y); return 1 - np.sum(p**2)

    def _best_split(self, X, y):
        best_gain, best_feat, best_thresh = -1, None, None
        g_parent = self._gini(y); n = len(y)
        feats = (np.random.choice(X.shape[1], self.max_features, replace=False)
                 if self.max_features else range(X.shape[1]))
        for feat in feats:
            for thresh in np.unique(X[:, feat]):
                left = y[X[:, feat] <= thresh]; right = y[X[:, feat] > thresh]
                if len(left) == 0 or len(right) == 0: continue
                gain = g_parent - (len(left)/n)*self._gini(left) - (len(right)/n)*self._gini(right)
                if gain > best_gain: best_gain, best_feat, best_thresh = gain, feat, thresh
        return best_feat, best_thresh

    def _grow(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split or len(np.unique(y)) == 1:
            return Node(value=np.bincount(y).argmax())
        feat, thresh = self._best_split(X, y)
        if feat is None: return Node(value=np.bincount(y).argmax())
        mask = X[:, feat] <= thresh
        return Node(feat, thresh, self._grow(X[mask], y[mask], depth+1),
                    self._grow(X[~mask], y[~mask], depth+1))

    def _traverse(self, x, node):
        if node.value is not None: return node.value
        return self._traverse(x, node.left if x[node.feature] <= node.threshold else node.right)


class RandomForest:
    def __init__(self, n_trees=20, max_depth=5, max_features='sqrt'):
        self.n_trees = n_trees; self.max_depth = max_depth
        self.max_features = max_features; self.trees = []

    def fit(self, X, y):
        n, d = X.shape
        n_feat = max(1, int(np.sqrt(d)) if self.max_features == 'sqrt' else d)
        for _ in range(self.n_trees):
            idx = np.random.choice(n, n, replace=True)  # bootstrap
            tree = DecisionTree(max_depth=self.max_depth, max_features=n_feat)
            tree.fit(X[idx], y[idx])
            self.trees.append(tree)

    def predict(self, X):
        preds = np.array([t.predict(X) for t in self.trees])  # (n_trees, n)
        return np.apply_along_axis(lambda col: np.bincount(col).argmax(), 0, preds)


X, y = load_iris(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForest(n_trees=20, max_depth=4)
rf.fit(X_tr, y_tr)
print(f"From-scratch RF accuracy: {np.mean(rf.predict(X_te) == y_te):.4f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import matplotlib.pyplot as plt

clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42).fit(X_tr, y_tr)
print(f"sklearn RF accuracy: {clf.score(X_te, y_te):.4f}")

# Feature importance
importances = clf.feature_importances_
plt.figure(figsize=(7, 4))
plt.bar(load_iris().feature_names, importances)
plt.title('Feature Importances (sklearn RF)')
plt.ylabel('Importance'); plt.tight_layout(); plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Variance vs n_trees
accs = []
for n in range(1, 51):
    rf_n = RandomForestClassifier(n_estimators=n, max_depth=4, random_state=42).fit(X_tr, y_tr)
    accs.append(rf_n.score(X_te, y_te))

plt.plot(range(1, 51), accs)
plt.xlabel('n_estimators'); plt.ylabel('Test Accuracy')
plt.title('Accuracy vs Number of Trees'); plt.tight_layout(); plt.show()

## Exercises

1. **Vary n_trees:** Plot test accuracy and standard deviation (over 5 random seeds) vs n_trees. Observe diminishing returns.
2. **OOB scoring:** Track which samples were not in each bootstrap. Use those samples to estimate OOB accuracy.
3. **max_features tuning:** Compare accuracy with max_features = 1, √d, d/2, d on a larger dataset.

## Summary

- Bagging reduces variance by averaging uncorrelated trees
- Feature subsampling de-correlates trees further — critical for the variance reduction
- Returns on additional trees diminish quickly — 50–200 trees usually sufficient

**Next:** [Neural Net From Scratch](neural-net-from-scratch.ipynb)